# Phase 14: YOLOv8 Leaf Detector Model Export to ONNX

This notebook loads your custom trained YOLOv8 leaf detection weights (`leaf_detector.pt`) and exports the model to **ONNX format (`leaf_detector.onnx`)** using Ultralytics' native compiler utilities.

In [1]:
# Setup paths and imports
from pathlib import Path
import sys
import shutil
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [2]:
# Load trained detector weights and compile to ONNX
weights_path = utils.paths.DETECTOR_MODEL_DIR / "leaf_detector.pt"
export_dest = utils.paths.PROJECT_ROOT / "exports" / "onnx" / "leaf_detector.onnx"
export_dest.parent.mkdir(parents=True, exist_ok=True)

if weights_path.exists():
    print("🚀 Loading YOLO model...")
    model = YOLO(str(weights_path))
    
    print("🚀 Starting YOLO-to-ONNX compilation...")
    onnx_file = model.export(format="onnx", imgsz=416)
    
    # Copy compile result to exports folder
    shutil.copy(onnx_file, str(export_dest))
    print("✅ YOLO leaf detector successfully compiled to:", export_dest)
else:
    print("⚠️ Warning: Custom leaf detector weights leaf_detector.pt not found on disk.")

🚀 Loading YOLO model...
🚀 Starting YOLO-to-ONNX compilation...
Ultralytics 8.4.102  Python-3.13.9 torch-2.11.0+cu128 CPU (12th Gen Intel Core i5-12450HX)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'O:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 5, 3549) (5.9 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.8s, saved as 'O:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.onnx' (11.6 MB)

Export complete (3.6s)
Results saved to O:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.onnx
Predict:         yolo predict task=detect model=O:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.onnx imgsz=416 
Validate:        yolo val task=detect model=O:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.onnx imgsz=416 data=O:\Hackthons\KrishiOS\ai\datasets\detection\plantdoc\data_coll

### Leaf Detector ONNX Model Specifications

*   **Input Tensor**: `images` of shape `[1, 3, 416, 416]` normalized to `[0.0, 1.0]` (Float32).
*   **Output Tensor**: `output0` of shape `[1, 5, 3549]` (representing center-x, center-y, width, height, and class confidence score for the 3549 bounding boxes).
*   **NMS Requirements**: Post-processing must apply Non-Maximum Suppression (NMS) with an intersection-over-union (IoU) threshold of `0.45` and class confidence threshold of `0.25` on the device CPU/GPU client layer.